# FX40 spectroscopy loader

Loads TIF files from the NX40 using their provided code and converts them to a dataframe

Make sure TIFF_Read_Write.py is in the same folder

TIFF_Read_Write.py found [here](https://spmdevhub.com/snippets/reading-and-writing-tiff-files/) on 23-09-2026

In [1]:
import os
import numpy as np
from pspylib.tiff.reader import TiffReader
import pandas as pd
from tqdm.auto import tqdm
import matplotlib.pyplot as plt
from TIFF_Read_Write import Read_tiff

g_0 = 7.748e-5

## Load with no processing

In [6]:
dataFolder = r"C:\Users\newsonj\OneDrive - Lancaster University\Documents\PhD\Electrospray\Fused Porphyrin\fP2\2026-10-01 new 100ugml 1hr\2026-10-02 JN fP2\CAFM\Memristor script test\Calibration\-2V - 2V"

if dataFolder[-1] != "/":
    dataFolder = dataFolder + "/"

print(dataFolder)

results = []

files = sorted(
    f for f in os.listdir(dataFolder)
    if os.path.isfile(os.path.join(dataFolder, f)) and f.lower().endswith('.tiff')
)

reader = Read_tiff()

for file in tqdm(files):
    spectDict, spectTiff = reader(os.path.join(dataFolder,file))
    v = spectDict['SPECT_DATA'][0][0]
    j = spectDict['SPECT_DATA'][0][1]
    g = np.gradient(j*1e-9,v)
    logg = np.log10(np.abs(g/g_0))
    logj = np.log10(np.abs(j*1e-9))

    results.append({
        "filename": file,
        "Bias": v.tolist(),
        "Current": (j).tolist(), 
        "dIdV": g.tolist(),
        "Log(G/G0)": logg.tolist(),
        "Log(Current)": logj.tolist()
    })

processedFolder = dataFolder + r'\Processed'
if not os.path.exists(processedFolder):
    os.makedirs(processedFolder)
df_all = pd.DataFrame(results)
print(df_all.head())
df_all.to_pickle(dataFolder + r'\Processed\FX40_Spectroscopy_Data.pkl')

print(f"Loaded {len(os.listdir(dataFolder))} ramp files")


C:\Users\newsonj\OneDrive - Lancaster University\Documents\PhD\Electrospray\Fused Porphyrin\fP2\2026-10-01 new 100ugml 1hr\2026-10-02 JN fP2\CAFM\Memristor script test\Calibration\-2V - 2V/


  0%|          | 0/600 [00:00<?, ?it/s]

C:\Users\newsonj\AppData\Local\Temp\ipykernel_33384\1358451495.py:22: RuntimeWarning: divide by zero encountered in log10
  logg = np.log10(np.abs(g/g_0))


               filename                                               Bias  \
0  iv_ 261005_001_.tiff  [0.0009598955512046814, 0.003919829614460468, ...   
1  iv_ 261005_002_.tiff  [0.0009598955512046814, 0.003919829614460468, ...   
2  iv_ 261005_003_.tiff  [0.0009598955512046814, 0.003919829614460468, ...   
3  iv_ 261005_004_.tiff  [0.0009598955512046814, 0.003919829614460468, ...   
4  iv_ 261005_005_.tiff  [0.0009598955512046814, 0.003919829614460468, ...   

                                             Current  \
0  [-0.006467000115662813, 0.0028927915263921022,...   
1  [0.010399251244962215, 0.0047740633599460125, ...   
2  [0.011442332528531551, -0.013805822469294071, ...   
3  [-0.025903701782226562, 0.01697438955307007, -...   
4  [-0.019701093435287476, 0.007642536889761686, ...   

                                                dIdV  \
0  [3.1621622110593875e-09, 2.08995780931498e-09,...   
1  [-1.9004436466496026e-09, -3.284527401987041e-...   
2  [-8.529972106896819e-09

## load and filter data

In [ ]:
dataFolder = r"C:\Users\newsonj\OneDrive - Lancaster University\Documents\PhD\Electrospray\Fused Porphyrin\fP2\2026-10-01 new 100ugml 1hr\2026-10-02 JN fP2\CAFM\Memristor script test\Calibration\-2V - 2V"

if dataFolder[-1] != "/":
    dataFolder = dataFolder + "/"

print(dataFolder)

SCBiasRange = [0,1] # define the bias range for filtering short circuits
OCBiasRange = [0.8,1.2] # define the bias range for filtering open circuits

maxCurrent = 9900 # max current in nA for filtering short circuits
minCurrent = 0.5 # min current in nA for filtering open circuits

results = []

files = sorted(
    f for f in os.listdir(dataFolder)
    if os.path.isfile(os.path.join(dataFolder, f)) and f.lower().endswith('.tiff')
)

reader = Read_tiff()

for file in tqdm(files):
    spectDict, spectTiff = reader(os.path.join(dataFolder,file))
    v = spectDict['SPECT_DATA'][0][0]
    j = spectDict['SPECT_DATA'][0][1]

    if any((SCBiasRange[0] < v[i] < SCBiasRange[1] and j[i] > maxCurrent) for i in range(len(v))):
        continue  # Skip this file if it meets the short circuit condition

    if any((OCBiasRange[0] < v[i] < OCBiasRange[1] and j[i] < minCurrent) for i in range(len(v))):
        continue  # Skip this file if it meets the open circuit condition

    g = np.gradient(j*1e-9,v)
    logg = np.log10(np.abs(g/g_0))
    logj = np.log10(np.abs(j*1e-9))

    results.append({
        "filename": file,
        "Bias": v.tolist(),
        "Current": (j).tolist(), 
        "dIdV": g.tolist(),
        "Log(G/G0)": logg.tolist(),
        "Log(Current)": logj.tolist()
    })

processedFolder = dataFolder + r'\Processed'
if not os.path.exists(processedFolder):
    os.makedirs(processedFolder)
df_all = pd.DataFrame(results)
print(df_all.head())
df_all.to_pickle(dataFolder + r'\Processed\FX40_Spectroscopy_Data_Filtered.pkl')

print(f"Loaded {len(os.listdir(dataFolder))} ramp files")


## Load and subtract a reference average

In [22]:
calibrationFile = r"C:\Users\newsonj\OneDrive - Lancaster University\Documents\PhD\Electrospray\Testing\2026-09-23 Memristor testing\Force dependent IV\Calibration\Processed\FX40_Spectroscopy_Data.pkl"
calib = pd.read_pickle(calibrationFile)
currentAvg = np.nanmean(np.array(calib["Current"].tolist()),axis=0)

dataFolder = r"C:\Users\newsonj\OneDrive - Lancaster University\Documents\PhD\Electrospray\Testing\2026-09-23 Memristor testing\Force dependent IV\0.5V Area 2"

if dataFolder[-1] != "/":
    dataFolder = dataFolder + "/"

print(dataFolder)

results = []

files = sorted(
    f for f in os.listdir(dataFolder)
    if os.path.isfile(os.path.join(dataFolder, f)) and f.lower().endswith('.tiff')
)

reader = Read_tiff()

for file in tqdm(files):
    spectDict, spectTiff = reader(os.path.join(dataFolder,file))
    v = spectDict['SPECT_DATA'][0][0]
    j = spectDict['SPECT_DATA'][0][1]-currentAvg
    g = np.gradient(j*1e-9,v)
    logg = np.log10(np.abs(g/g_0))
    logj = np.log10(np.abs(j*1e-9))

    results.append({
        "filename": file,
        "Bias": v.tolist(),
        "Current": (j).tolist(), 
        "dIdV": g.tolist(),
        "Log(G/G0)": logg.tolist(),
        "Log(Current)": logj.tolist()
    })

processedFolder = dataFolder + r'\Processed'
if not os.path.exists(processedFolder):
    os.makedirs(processedFolder)
df_all = pd.DataFrame(results)
print(df_all.head())
df_all.to_pickle(dataFolder + r'\Processed\FX40_Spectroscopy_Data_corrected.pkl')

print(f"Loaded {len(os.listdir(dataFolder))} ramp files")


C:\Users\newsonj\OneDrive - Lancaster University\Documents\PhD\Electrospray\Testing\2026-09-23 Memristor testing\Force dependent IV\0.5V Area 2/


  0%|          | 0/501 [00:00<?, ?it/s]

                         filename  \
0  fP2 IV test_260923_3419__.tiff   
1  fP2 IV test_260923_3420__.tiff   
2  fP2 IV test_260923_3421__.tiff   
3  fP2 IV test_260923_3422__.tiff   
4  fP2 IV test_260923_3423__.tiff   

                                                Bias  \
0  [0.0019198935478925705, 0.007839822210371494, ...   
1  [0.0019198935478925705, 0.007839822210371494, ...   
2  [0.0019198935478925705, 0.007839822210371494, ...   
3  [0.0019198935478925705, 0.007839822210371494, ...   
4  [0.0019198935478925705, 0.007839822210371494, ...   

                                             Current  \
0  [-0.005212991964071989, -0.00556654962484673, ...   
1  [-0.03524814499542117, -0.003973988022254262, ...   
2  [0.00029112445190548897, 0.007322954807831492,...   
3  [-0.031047880183905363, 0.002657028709007945, ...   
4  [0.01807938562706113, 0.007155316744400706, 0....   

                                                dIdV  \
0  [-5.972329751465152e-11, 2.069723256581785e-

# Memristor script loader

In [31]:
baseDir = r"C:\Users\newsonj\OneDrive - Lancaster University\Documents\PhD\Electrospray\Fused Porphyrin\fP2\2026-10-01 new 100ugml 1hr\2026-10-02 JN fP2\CAFM\Memristor script test"

sequence = ["WIDE", "WRITE", "READW", "READW", "READW", "READW", "ERASE", "READE", "READE", "READE", "READE", "WIDE"]

calibrationFiles = [r"C:\Users\newsonj\OneDrive - Lancaster University\Documents\PhD\Electrospray\Fused Porphyrin\fP2\2026-10-01 new 100ugml 1hr\2026-10-02 JN fP2\CAFM\Calibration\-2 - 2V\Processed\FX40_Spectroscopy_Data.pkl",
                    r"C:\Users\newsonj\OneDrive - Lancaster University\Documents\PhD\Electrospray\Fused Porphyrin\fP2\2026-10-01 new 100ugml 1hr\2026-10-02 JN fP2\CAFM\Calibration\0 - 2V\Processed\FX40_Spectroscopy_Data.pkl",
                    r"C:\Users\newsonj\OneDrive - Lancaster University\Documents\PhD\Electrospray\Fused Porphyrin\fP2\2026-10-01 new 100ugml 1hr\2026-10-02 JN fP2\CAFM\Calibration\0 - 0.5V\Processed\FX40_Spectroscopy_Data.pkl",
                    r"C:\Users\newsonj\OneDrive - Lancaster University\Documents\PhD\Electrospray\Fused Porphyrin\fP2\2026-10-01 new 100ugml 1hr\2026-10-02 JN fP2\CAFM\Calibration\0 - 0.5V\Processed\FX40_Spectroscopy_Data.pkl",
                    r"C:\Users\newsonj\OneDrive - Lancaster University\Documents\PhD\Electrospray\Fused Porphyrin\fP2\2026-10-01 new 100ugml 1hr\2026-10-02 JN fP2\CAFM\Calibration\0 - -2V\Processed\FX40_Spectroscopy_Data.pkl"]

calib = [pd.read_pickle(f) for f in calibrationFiles]
currentAvgs = {"WIDE": np.nanmean(np.array(calib[0]["Current"].tolist()),axis=0),
               "WRITE": np.nanmean(np.array(calib[1]["Current"].tolist()),axis=0),
               "READW": np.nanmean(np.array(calib[2]["Current"].tolist()),axis=0),
               "READE": np.nanmean(np.array(calib[3]["Current"].tolist()),axis=0),
               "ERASE": np.nanmean(np.array(calib[4]["Current"].tolist()),axis=0)}

folders = sorted(
    f for f in os.listdir(baseDir)
    if os.path.isdir(os.path.join(baseDir, f))
)

for folder in tqdm(folders):
    files = [f for f in os.listdir(os.path.join(baseDir, folder)) if f.endswith('.tiff')]
    files.sort(key=lambda f: int(os.path.splitext(f)[0].rstrip("_").rsplit("_", 1)[-1]))
    #print(files)

    filesSplit = [files[i:i+len(sequence)] for i in range(0, len(files), len(sequence))]
    print(folder + ": " + str(len(filesSplit)) + " loops")
    #print(filesSplit)

    data = []

    for group in filesSplit:
        valid = True
        groupData = []
        for i, file in enumerate(group):
            spectDict, spectTiff = reader(os.path.join(baseDir, folder, file))
            v = spectDict['SPECT_DATA'][0][0]
            j = spectDict['SPECT_DATA'][0][1]-currentAvgs[sequence[i]]
            g = np.gradient(j*1e-9,v)
            logg = np.log10(np.abs(g/g_0))
            logj = np.log10(np.abs(j*1e-9))
            if np.max(np.abs(j))>9900:
                valid = False
                print(f"Invalid data in {file}, skipping loop {group}")
                break
            groupData.append({
                "filename": file,
                "Type": sequence[i],
                "Bias": v.tolist(),
                "Current": (j).tolist(), 
                "dIdV": g.tolist(),
                "Log(G/G0)": logg.tolist(),
                "Log(Current)": logj.tolist()
            })
        if valid:
            data.extend(groupData)

    if not os.path.exists(os.path.join(baseDir, folder, 'Processed')):
        os.makedirs(os.path.join(baseDir, folder, 'Processed'))
    df_all = pd.DataFrame(data)
    df_all.to_pickle(os.path.join(baseDir, folder, 'Processed', 'FX40_Spectroscopy_Data_corrected.pkl'))




  0%|          | 0/31 [00:00<?, ?it/s]

0: 1 loops
Invalid data in iv_ 261002_3390_.tiff, skipping loop ['iv_ 261002_3390_.tiff', 'iv_ 261002_3391_.tiff', 'iv_ 261002_3392_.tiff', 'iv_ 261002_3393_.tiff', 'iv_ 261002_3394_.tiff', 'iv_ 261002_3395_.tiff', 'iv_ 261002_3396_.tiff', 'iv_ 261002_3397_.tiff', 'iv_ 261002_3398_.tiff', 'iv_ 261002_3399_.tiff', 'iv_ 261002_3400_.tiff', 'iv_ 261002_3401_.tiff']
1: 17 loops
Invalid data in iv_ 261002_3595_.tiff, skipping loop ['iv_ 261002_3594_.tiff', 'iv_ 261002_3595_.tiff', 'iv_ 261002_3596_.tiff', 'iv_ 261002_3597_.tiff', 'iv_ 261002_3598_.tiff', 'iv_ 261002_3599_.tiff', 'iv_ 261002_3600_.tiff', 'iv_ 261002_3601_.tiff', 'iv_ 261002_3602_.tiff', 'iv_ 261002_3603_.tiff', 'iv_ 261002_3604_.tiff', 'iv_ 261002_3605_.tiff']
10: 1 loops
Invalid data in iv_ 261002_5517_.tiff, skipping loop ['iv_ 261002_5517_.tiff', 'iv_ 261002_5518_.tiff', 'iv_ 261002_5519_.tiff', 'iv_ 261002_5520_.tiff', 'iv_ 261002_5521_.tiff', 'iv_ 261002_5522_.tiff', 'iv_ 261002_5523_.tiff', 'iv_ 261002_5524_.tiff', 'iv